In [ ]:
#Importacion de librerias
import requests
from sklearn.preprocessing import OrdinalEncoder
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedShuffleSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve, precision_recall_curve
import xgboost as xgb
import joblib  # Para guardar y cargar el modelo
import shap    # Para la interpretabilidad del modelo


In [ ]:
# Configura los detalles de la conexión desde variables de entorno.
# Copia .env.example a .env y define los valores. Nunca escribas credenciales aquí.
import os

host     = os.getenv('DB_HOST')
port     = os.getenv('DB_PORT', '1433')
database = os.getenv('DB_NAME')

if not all([host, database]):
    raise RuntimeError(
        "Faltan variables de entorno: define DB_HOST y DB_NAME. Ver .env.example"
    )

try:
    # Crea la cadena de conexión
    url = (
        'mssql+pyodbc://@{host}:{port}/{db}'
        '?trusted_connection=yes&driver=SQL+Server'
    ).format(host=host, port=port, db=database)

    # Crear conexion con base de datos
    engine = create_engine(url)
    print("Conexion a la base de datos realizada")

except Exception as e:
    print('Error:', e)


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


SELECT 
    Identificacion,
    Periodo,
	Reintegro,
    Periodo_Siguiente,
    EX_Periodo_Siguiente_Normal,
    nuevo_periodo,
    Status,
	Tipo_Salto,
    Estado_Alumno,
    Tipo_estado_alumno,
    Modalidad,
    Programa,
    Semestre_SINU,
	ciclo,
	año,
    AÑO_MEN,
	Estado_Alumno,
    anio_grado,
    Genero,
    Estado_Pago,
    RANGO_EDAD,
    RANGO_SALARIO,
    ESTA_TRABAJANDO,
    METODO_FINANCIAMIENTO,
    ZONA_RESIDENCIA,
    REGIMEN_SISTEMA_SALUD,
    PERTENECE_GRUPO_ETNICO,
    GRUPO_ETNICO,
    TIENE_DISCAPACIDAD,
    DISCAPACIDAD,
    LGBTIQ,
	ciudad,
	localidad_normalizada
FROM academico.historial_academico


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        historial_academico = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


SELECT 
Identificacion, 
barrio, 
departamento, 
pais
FROM geo.ubicacion_estudiante


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        localización = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


SELECT
Periodo,
NOMBRE_GRUPO, 
IDENTIFICACION as Identificacion, 
NOMBRE_CONCEPTO, 
NOMBRE_CAUSA_NOTA
FROM becas.descuentos_beca


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        becas = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


select 
IDENTIFICACION as Identificacion, 
[MATERIAS INSCRITAS], 
[MATERIAS APROBADAS], 
COD_PERIODO as Periodo, 
Porcentaje_aprobacion
FROM academico.aprobacion_materias 


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        materias = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


select Periodo, Identificacion,TOTAL from 
financiera.cartera


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        finanza = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


select *
from plataforma.permanencia


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        permanencia = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

# Hago copia
df = historial_academico.copy()


In [ ]:

# Verifico Columnas y Filas
df.shape


In [ ]:

# Ahora haz el merge correctamente
df1 = df.merge(
    localización,
    on='Identificacion',
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df1.shape


In [ ]:

# Borro duplicados de identificación y periodo porque me interesa solammente cursado en ultimo periodo
becas = becas[~becas.duplicated(subset=['Identificacion', 'Periodo'], keep='last')]


In [ ]:

# Ahora haz el merge correctamente
df2 = df1.merge(
    becas,
    on=['Identificacion','Periodo'],
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df2.shape


In [ ]:

# Ahora haz el merge correctamente
df3 = df2.merge(
    materias,
    on=['Identificacion', 'Periodo'],
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df3.shape


In [ ]:

# Borro duplicados de identificación y periodo porque me interesa solammente cursado en ultimo periodo
finanza = finanza[~finanza.duplicated(subset=['Identificacion', 'Periodo'], keep='last')]


In [ ]:

# Ahora haz el merge correctamente
df4 = df3.merge(
    finanza,
    on=['Identificacion', 'Periodo'],
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df4.shape


In [ ]:

# Paso 0: Lista de periodos a eliminar
periodos_a_excluir = [
    '24I01', '24I02', '24I03', '24I04', '24I05', '24I06',
    '24I10', '24I11', '24I12', '24I13',
    '25I01', '25I02', '25I03', '25I04', '25I11', '25I12', '25I13'
]


In [ ]:

# Paso 1: Eliminar los periodos que no se desean
permanencia = permanencia[~permanencia['cod_periodo'].isin(periodos_a_excluir)]


In [ ]:

# Paso 2: Obtener el último periodo (cod_periodo) por cada estudiante
ultimo_periodo = permanencia.groupby('num_identificacion')['cod_periodo'].max().reset_index()
ultimo_periodo.rename(columns={'cod_periodo': 'ultimo_cod_periodo'}, inplace=True)


In [ ]:

# Paso 3: Unir esta información al DataFrame original
permanenciaa = permanencia.merge(ultimo_periodo, on='num_identificacion', how='left')


In [ ]:

# Paso 4: Filtrar solo las filas del último periodo por estudiante
permanenciaa = permanenciaa[permanenciaa['cod_periodo'] == permanenciaa['ultimo_cod_periodo']]


In [ ]:

# Paso 5: Eliminar duplicados (por si un curso se repite por error)
permanenciaa = permanenciaa.drop_duplicates(subset=['num_identificacion', 'curso', 'cod_periodo'])


In [ ]:

# Paso 6: Eliminar la columna auxiliar que ya no necesitamos
permanenciaa = permanenciaa.drop(columns=['ultimo_cod_periodo'])


In [ ]:

# Paso 7: Ordenar por estudiante y curso
permanenciaa = permanenciaa.sort_values(by=['num_identificacion', 'curso'])


In [ ]:

# Paso 8: Reiniciar índices
permanenciaa = permanenciaa.reset_index(drop=True)


In [ ]:

# Resultado final:
permanenciaa


In [ ]:

# Verifico Columnas
permanenciaa.columns


In [ ]:

# Renombro para que me cruzen las llaves
permanenciaa = permanenciaa.rename(columns={
    'num_identificacion': 'Identificacion',
    'cod_periodo': 'Periodo'
})


In [ ]:

# Verifico Columnas
permanenciaa.columns


In [ ]:

# Borro las variables que no necesito
permanenciaa = permanenciaa.drop(columns=[
    'SEMESTRE', 'NOM_UNIDAD', 'NOM_DEPENDENCIA', 'CRE_PROGRAMA',
    'CREDITOS_INSCRITOS', 'Materias_BU', 'Materias_B1', 'Materias_B2',
    'nivelado_en_materias', 'itemtype', 'itemmodule', 'itemname',
    'nombrecategoria', 'peso', 'finalgrade', 'estadoactividad',
    'estudiantes', 'curso', '#ASISTENCIAS'  
    
])


In [ ]:

# Verifico Columnas
permanenciaa.columns


In [ ]:

# Borro duplicados de identificación y periodo porque me interesa solammente cursado en ultimo periodo
permanenciaa = permanenciaa.drop_duplicates(subset=['Identificacion', 'Periodo'])


In [ ]:

# Verifico Columnas y Filas
df4.shape


In [ ]:

# Ahora haz el merge correctamente

df5 = df4.merge(
    permanenciaa,
    on=['Identificacion', 'Periodo'],
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df5.shape


In [ ]:
# DF final

df5


In [ ]:
# Verifico las columnas del DataFrame final

print(df5.columns.tolist())


In [ ]:
# Verifico las columnas del DataFrame final

print(df5.columns.tolist())

In [ ]:
df5.columns = df5.columns.str.strip().str.upper().str.replace(' ', '_')

In [ ]:
# Prototipo si quiero filtrar por Periodo y borro duplicados de identificación

#df5=df5.drop_duplicates(['Identificacion'])


#df5  = df5[df5['Periodo']=='2025B']

# Paso 0: Lista de periodos a eliminar
#año_a_excluir = [
#    '2024', '2025'
#]

# Paso 1: Eliminar los años que no se desean
#df5 = df5[~df5['AÑO'].isin(año_a_excluir)]

In [ ]:
## Verifico si hay valores nulos

df5.isnull().sum()


In [ ]:
# Verifico si hay valores nulos en la columna 'ASISTENCIA'
df5['ASISTENCIA'].value_counts(dropna=False)


In [ ]:
#Covversion de columnas sin espacios y mayusculas

df5.columns = df5.columns.str.strip().str.upper().str.replace(' ', '_')

In [ ]:
# ...imprime la lista para estar seguro de los nombres correctos
print(df5.columns.tolist())

In [ ]:
# Prototipo si quiero crear una columna de target de deserción

#df5['TARGET_DESERCION'] = df5['STATUS'].apply(
#    lambda x: 1 if str(x).strip().lower() == 'desercion' else 0
#)

In [ ]:
# Mi columna objetivo es 'TARGET_DESERCION' y ha sido creada para identificar si un estudiante ha desertado o no y convertido ha binario.


df5['TARGET_DESERCION'] = df5['STATUS'].apply(
    lambda x: 1 if str(x).strip().lower() == 'desercion' else 0
)

In [ ]:
# Columna objetivo
col_objetivo = 'TARGET_DESERCION'


In [ ]:
# Verifico si hay valores nulos

df5.isnull().sum()

In [ ]:
# =============================================================================
# PASO 1: CARGA Y LIMPIEZA DE DATOS
# =============================================================================
# ... (Aquí va todo tu código de conexión a la BD, extracción de las 6 tablas y merge hasta tener 'df5') ...
# Asumimos que 'df5' es el DataFrame resultante de todas tus fusiones.

# Estandarización de nombres de columnas
df5.columns = df5.columns.str.strip().str.upper().str.replace(' ', '_')
df5.rename(columns={'MATERIAS_INSCRITAS': 'MATERIAS_INSCRITAS', 'MATERIAS_APROBADAS': 'MATERIAS_APROBADAS'}, inplace=True, errors='ignore')


In [ ]:

# =============================================================================
# PASO 2: INGENIERÍA DE CARACTERÍSTICAS
# =============================================================================
print("\nAplicando ingeniería de características...")
df5['FLAG_NO_APROBO_NADA'] = np.where(df5['PORCENTAJE_APROBACION'] == 0, 1, 0)
df5['PORCENTAJE_APROBACION'] = df5['PORCENTAJE_APROBACION'].replace(0, np.nan)
print("Nueva característica 'FLAG_NO_APROBO_NADA' creada.")


In [ ]:
df5

In [ ]:

# =============================================================================
# PASO 3: SELECCIÓN DE CARACTERÍSTICAS Y PREPROCESAMIENTO
# =============================================================================

# Identificadores (no se usan como features, pero se conservan)
identificadores = ['IDENTIFICACION', 'PERIODO', 'AÑO']


In [ ]:

# Variable target
target = 'TARGET_DESERCION'


In [ ]:

columnas_numericas = [
    'SEMESTRE_SINU', 'MATERIAS_INSCRITAS', 'MATERIAS_APROBADAS',
    'PORCENTAJE_APROBACION', 'TOTAL', 'FLAG_NO_APROBO_NADA'
]


In [ ]:

columnas_categoricas = [
    'TIPO_SALTO', 'MODALIDAD', 'GENERO', 'ESTADO_PAGO', 'RANGO_EDAD',
    'RANGO_SALARIO', 'ESTA_TRABAJANDO', 'METODO_FINANCIAMIENTO',
    'ZONA_RESIDENCIA', 'REGIMEN_SISTEMA_SALUD', 'ASISTENCIA'
]


In [ ]:

# Variable target
target = 'TARGET_DESERCION'

features = columnas_numericas + columnas_categoricas


In [ ]:

# Seleccionar solo identificadores + features + target
columnas_ml = identificadores + features + [target]


In [ ]:

# Crear un DataFrame reducido para ML
df_ml = df5[columnas_ml]


In [ ]:

# Transformaciones
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
    # sin OneHotEncoder para mantener columnas iguales
])

# Aplicar imputación a numéricas
df_num = pd.DataFrame(
    numeric_transformer.fit_transform(df5[columnas_numericas]),
    columns=columnas_numericas,
    index=df5.index
)

# Aplicar imputación a categóricas
df_cat = pd.DataFrame(
    categorical_transformer.fit_transform(df5[columnas_categoricas]),
    columns=columnas_categoricas,
    index=df5.index
)

# Reconstruir el DataFrame final con misma forma
df_final_ml = pd.concat([
    df5[identificadores],   # IDENTIFICACION y AÑO originales
    df_num,                 # numéricas imputadas
    df_cat,                 # categóricas imputadas
    df5[[target]]           # target intacto
], axis=1
)

In [ ]:
# Comparar identificadores originales vs procesados
comparacion = (df5[identificadores].reset_index(drop=True) == 
               df_final_ml[identificadores].reset_index(drop=True))

# Mostrar si todas las filas son iguales
print("¿IDENTIFICACION coincide en todas las filas?:", comparacion['IDENTIFICACION'].all())
print("¿AÑO coincide en todas las filas?:", comparacion['AÑO'].all())

# Si quieres ver las primeras diferencias (si existieran)
diferencias = df5[identificadores].reset_index(drop=True).compare(
    df_final_ml[identificadores].reset_index(drop=True)
)
print("Diferencias encontradas:")
print(diferencias.head())

In [ ]:
df_final_ml

In [ ]:
df_final_ml['AÑO'].value_counts()

In [ ]:
df_final_ml.shape

In [ ]:
df_ml.shape

In [ ]:
df_final_ml
# Guardar en Excel
df_final_ml.to_excel("Despliegue2024.xlsx", index=False)  # index=False para no guardar la columna de índices

In [ ]:
df_desercion = df_final_ml.copy()

In [ ]:
df_desercion.columns

In [ ]:
df_desercion.shape

In [ ]:
df_desercion = df_desercion[~df_desercion['AÑO'].astype(str).isin(['2024', '2025'])]

In [ ]:
df_desercion.shape

### entrnamiebto


In [ ]:
# X = todo menos identificadores y target
X = df_desercion.drop(columns=identificadores + [target])

# y = solo la variable objetivo
y = df_desercion[target]

In [ ]:
X.columns

In [ ]:

# =============================================================================
# PASO 4: DIVISIÓN DE DATOS
# =============================================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"\nDatos divididos: {len(X_train)} para entrenamiento, {len(X_test)} para prueba.")


In [ ]:
# --- PREPROCESSOR PARA EL MODELO (con OneHot para categorías) ---
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), columnas_numericas),
        
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), columnas_categoricas)
    ],
    remainder='drop'
)

In [ ]:
X

In [ ]:

# =============================================================================
# PASO 5: OPTIMIZACIÓN DE HIPERPARÁMETROS (GRIDSEARCHCV)
# =============================================================================
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(random_state=42, eval_metric='logloss'))
])
splitter = StratifiedShuffleSplit(n_splits=1, train_size=0.15, random_state=42)
for train_index, _ in splitter.split(X_train, y_train):
    X_sample, y_sample = X_train.iloc[train_index], y_train.iloc[train_index]

param_grid_xgb = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 7],
    'classifier__learning_rate': [0.05, 0.1]
}
grid_search = GridSearchCV(estimator=xgb_pipeline, param_grid=param_grid_xgb, cv=3, scoring='f1', n_jobs=-1, verbose=2)
print("\nIniciando la búsqueda de los mejores hiperparámetros...")
grid_search.fit(X_sample, y_sample)
print("\nMejores parámetros encontrados:", grid_search.best_params_)


In [ ]:

# =============================================================================
# PASO 6: RE-ENTRENAMIENTO DEL MODELO FINAL
# =============================================================================
print("\nPreprocesando y aplicando SMOTE al set de entrenamiento completo...")
X_train_processed = preprocessor.fit_transform(X_train)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)
print("Balanceo con SMOTE completado.")

best_params = grid_search.best_params_
final_params = {key.replace('classifier__', ''): value for key, value in best_params.items()}
final_classifier = xgb.XGBClassifier(**final_params, random_state=42, eval_metric='logloss')

print("\nRe-entrenando el mejor clasificador con todos los datos balanceados...")
final_classifier.fit(X_train_resampled, y_train_resampled)
print("Re-entrenamiento completado.")

# 🚀 Nuevo: armar pipeline completo (preprocesador + clasificador)
pipeline_final = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', final_classifier)
])

# Guardar todo en un único archivo .pkl
joblib.dump(pipeline_final, 'modelo_finall.pkl')
print("✅ Pipeline completo guardado en 'modelo_finall.pkl'")

In [ ]:

# =============================================================================
# PASO 7: BÚSQUEDA AUTOMÁTICA DEL MEJOR UMBRAL
# =============================================================================
print("\nBuscando el mejor umbral de decisión...")
X_test_processed = preprocessor.transform(X_test)
y_proba = final_classifier.predict_proba(X_test_processed)[:, 1]
thresholds = np.arange(0.1, 0.6, 0.01)
f1_scores = [f1_score(y_test, (y_proba >= t).astype(int)) for t in thresholds]
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Mejor umbral encontrado para maximizar F1-Score: {best_threshold:.2f}")


In [ ]:

# =============================================================================
# PASO 8: ANÁLISIS COMPLETO DEL MODELO FINAL
# =============================================================================
y_pred_final = (y_proba >= best_threshold).astype(int)


In [ ]:

# --- Reporte de Clasificación y Matriz de Confusión ---
print(f"\n--- Reporte de Clasificación Final (con Umbral Óptimo de {best_threshold:.2f}) ---")
print(classification_report(y_test, y_pred_final))
cm = confusion_matrix(y_test, y_pred_final)
plt.figure(figsize=(8, 6)); sns.heatmap(cm, annot=True, fmt='d', cmap='viridis');
plt.title(f'Matriz de Confusión - Modelo Final (Umbral {best_threshold:.2f})'); plt.ylabel('Etiqueta Real'); plt.xlabel('Etiqueta Predicha'); plt.show()


In [ ]:

# --- Curva ROC y AUC ---
auc = roc_auc_score(y_test, y_proba); fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(8, 6)); plt.plot(fpr, tpr, label=f'XGBoost Optimizado (AUC = {auc:.4f})');
plt.plot([0, 1], [0, 1], 'r--'); plt.title('Curva ROC del Modelo Final');
plt.xlabel('Tasa de Falsos Positivos'); plt.ylabel('Tasa de Verdaderos Positivos'); plt.legend(); plt.grid(); plt.show()


In [ ]:

# --- Top 15 Variables Más Importantes ---
ohe_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(columnas_categoricas)
final_feature_names = columnas_numericas + list(ohe_feature_names)
importancia = pd.Series(final_classifier.feature_importances_, index=final_feature_names).sort_values(ascending=False)
plt.figure(figsize=(12, 8)); sns.barplot(x=importancia.head(15), y=importancia.head(15).index);
plt.title('Top 15 Variables más importantes - Modelo Final Optimizado'); plt.xlabel('Importancia'); plt.ylabel('Variable'); plt.show()


In [ ]:

# =============================================================================
# PASO 9: INTERPRETABILIDAD CON SHAP (VERSIÓN MODERNA)
# =============================================================================
import shap

print("\nCalculando valores SHAP con versión moderna...")

# --- Preparar datos para SHAP ---
# Convertir todo a float para evitar warnings
X_test_processed_df = pd.DataFrame(
    X_test_processed, 
    columns=final_feature_names
).astype(float)

# --- Crear el Explainer usando la última versión ---
explainer = shap.Explainer(final_classifier, X_test_processed_df)

# --- Calcular valores SHAP ---
shap_values = explainer(X_test_processed_df)

# --- Visualización global: importancia de las características ---
shap.summary_plot(shap_values.values, X_test_processed_df, plot_type='bar', show=False)
plt.title("Importancia global de las características (SHAP)")
plt.show()

# --- Visualización global: impacto positivo/negativo de cada feature ---
shap.summary_plot(shap_values.values, X_test_processed_df)
plt.title("Impacto de las características en la predicción (SHAP)")
plt.show()

# --- Ejemplo de interpretabilidad local para las primeras 5 predicciones ---
for i in range(5):
    print(f"\nExplicación SHAP para el estudiante {i+1}:")
    shap.waterfall_plot(shap_values[i])


In [ ]:
# =============================================================================
# PASO 10: GUARDADO DEL PIPELINE CON UMBRAL AUTOMÁTICO Y FUNCIÓN DE PREDICCIÓN
# =============================================================================

# --- Guardar el mejor umbral dentro del pipeline ---
pipeline_final.best_threshold = best_threshold
joblib.dump(pipeline_final, 'modelo_final.pkl')
print("✅ Pipeline completo guardado con el mejor umbral en 'modelo_final.pkl'")

# --- Función de predicción usando umbral automático ---
def predecir_desercion_auto(nuevos_datos, model_path='modelo_final.pkl', umbral=None):
    """
    Predice la deserción para nuevos datos usando el pipeline completo guardado.
    Si no se pasa un umbral, se usa el mejor umbral almacenado dentro del pipeline.

    :param nuevos_datos: DataFrame con las mismas columnas de entrenamiento (features, sin target ni identificadores).
    :param model_path: Ruta al pipeline guardado.
    :param umbral: Umbral de decisión opcional. Si es None, se usa pipeline.best_threshold.
    :return: DataFrame con columnas ['PROBABILIDAD_DESERCION', 'PREDICCION_DESERCION']
    """
    
    # Cargar pipeline completo
    pipeline = joblib.load(model_path)
    
    # Usar el umbral almacenado si no se proporciona
    if umbral is None:
        if hasattr(pipeline, 'best_threshold'):
            umbral = pipeline.best_threshold
        else:
            umbral = 0.5  # fallback
        print(f"Usando umbral automático: {umbral:.2f}")
    
    # Calcular probabilidades
    probabilidades = pipeline.predict_proba(nuevos_datos)[:, 1]
    
    # Aplicar umbral
    predicciones = (probabilidades >= umbral).astype(int)
    
    # Devolver DataFrame con probabilidades y predicciones
    resultado = pd.DataFrame({
        'PROBABILIDAD_DESERCION': probabilidades,
        'PREDICCION_DESERCION': predicciones
    })
    
    return resultado

# --- Ejemplo de uso ---
muestra_nuevos_datos = X_test.head(5)  # Tus nuevas filas
predicciones_ejemplo = predecir_desercion_auto(muestra_nuevos_datos)
print(predicciones_ejemplo)
